# LUAD Data Preprocessing Pipeline
Builds node features (HPA, GDC mutation/CNV), STRING PPI network, conventional network topology features, and GDA labels.


In [2]:
pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 16.1 MB/s eta 0:00:00 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
import igraph as ig  # pip install python-igraph
from collections import Counter
from node2vec import Node2Vec
import networkx as nx


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
12.8
True
NVIDIA GeForce RTX 5060


In [3]:
def remove_redun(el, verbose=False):
    if verbose:
        print("Original Size: ", len(el))

    el_new = el.iloc[:, 0:2].apply(sorted, axis=1)
    el_new = pd.DataFrame.from_dict(dict(zip(el_new.index, el_new.values))).T
    el_new = el_new.drop_duplicates()
    if verbose:
        postDrop = len(el_new)
        print("After Dropping Duplicates: ", len(el_new), "(-", len(el)-postDrop, ")")

    el_new = el_new.merge(el, left_on=[el_new.columns[0], el_new.columns[1]],
                          right_on=[el.columns[0], el.columns[1]])
    if verbose:
        print("After Merging: ", len(el_new), "(-", postDrop-len(el_new), ")")
        print()

    return el_new.iloc[:, 2:]


def map_IDs(el, gmap, verbose=False, dropNaNvalues=True):
    gp_map_f = gmap.set_index('#string_protein_id')['alias']
    el_converted = el.reset_index(drop=True)

    el_converted[el_converted.columns[0]] = el_converted[el_converted.columns[0]].map(gp_map_f)
    el_converted[el_converted.columns[1]] = el_converted[el_converted.columns[1]].map(gp_map_f)

    if verbose:
        print("NaN values per Column:",
              el_converted[el_converted.columns[0]].isna().sum(),
              el_converted[el_converted.columns[1]].isna().sum())

    if dropNaNvalues:
        el_converted = el_converted.dropna()
        if verbose:
            print("New edge list size:", len(el_converted), "( -", len(el)-len(el_converted), ")")

    return el_converted


def ImportSTRING():
    el_map = pd.read_csv(
        r"9606.protein.aliases.v12.0.txt",
        sep="\t"
    )
    el = pd.read_csv(
        r"9606.protein.links.v12.0_sc.txt",
        sep=" "
    )
    el_map = el_map.loc[el_map.source == 'Ensembl_gene']

    el = remove_redun(el, True)
    el = map_IDs(el, el_map, verbose=True)

    return el

In [4]:
def ImportDGN():
    dgn = pd.read_csv("C0279626_disease_gda_summary.tsv",sep="\t")
    dgn_dict = pd.read_csv(
        "/media/vvlab/Expansion1/akash/gcnescc/20th_July_repeat_GCN/8_07_26/gda_dictionary.csv",sep=",",
        index_col=None
    )

    score_threshold = 0.1  # no-op for ESCA
    ei_threshold    = 0.7

    dgn = dgn[['Gene_Symbol', 'Ei', 'Score']]
    dgn = dgn.loc[dgn['Score'] >= score_threshold]
    dgn = dgn.loc[dgn['Ei'] > ei_threshold]
    dgn.rename({'Score': 'gda_score'}, axis=1, inplace=True)
    dgn = dgn.merge(dgn_dict, on="Gene_Symbol").drop(['Gene_Symbol'], axis=1)
    dgn['gda_score'] = 1

    return dgn[['ensembl', 'gda_score']]

In [5]:
def ImportHPA():
    hpa = pd.read_csv(
        r"esophagus.tsv",
        sep='\t'
    ).drop_duplicates(subset='Gene')

    identifiers = ["Gene", "Ensembl"]
    discrete_features = [
        "Protein class", "Biological process", "Molecular function",
        "Disease involvement", "Subcellular location",
    ]

    hpa_features = hpa.loc[:, hpa.columns.isin(identifiers + discrete_features)]

    # normalise continuous features

    def explode(feature):
        return feature.apply(lambda x: x.replace(' ', '').split(','))

    hpa_clean = hpa.fillna('')
    for ft in discrete_features:
        hpa_clean[ft] = explode(hpa_clean[ft])

    protein_class        = hpa_clean["Protein class"].explode().unique()
    biological_process   = hpa_clean["Biological process"].explode().unique()
    molecular_function   = hpa_clean["Molecular function"].explode().unique()
    disease_involvement  = hpa_clean["Disease involvement"].explode().unique()
    subcellular_location = hpa_clean["Subcellular location"].explode().unique()

    GO_features = np.concatenate([
        protein_class, biological_process, molecular_function,
        disease_involvement, subcellular_location
    ])

    RowFeatures = pd.DataFrame(data=0, index=hpa_clean['Ensembl'], columns=GO_features)
    counter = 0
    for index, row in RowFeatures.iterrows():
        features = hpa_clean.iloc[counter][
            ['Protein class', 'Biological process', 'Molecular function',
             'Disease involvement', 'Subcellular location']
        ].to_list()
        flattened = [item for sublist in features for item in sublist if item]
        for t in flattened:
            row[t] = 1
        counter += 1

    # truncated SVD to 200 dims
    n_comp    = 200
    svd       = TruncatedSVD(n_components=n_comp)
    svdModel  = svd.fit(RowFeatures)
    visits_emb = svdModel.transform(RowFeatures)

    hpa_reduced = pd.DataFrame(data=visits_emb, index=RowFeatures.index).reset_index(names="Ensembl")

    hpa_final = hpa_reduced

    hpa_final.columns = ['hpa_' + str(col) for col in hpa_final.columns]
    hpa_final = hpa_final.rename({
        'hpa_Ensembl': 'ensembl',
    }, axis=1)

    return hpa_final

In [6]:
def ImportGDC():
    gdc = pd.read_csv("CNVs_ESCA.tsv", sep='\t')

    gdc = gdc.rename(columns={
        'gene_id': 'ensembl',
        'cohort_ssm_affected_cases_percentage': 'nih_ssm_in_cohort',
        'gdc_ssm_affected_cases_percentage':    'nih_ssm_across_gdc',
        'cohort_cnv_gain_cases_percentage':     'nih_cnv_gain',
        'num_mutations':                        'nih_tot_mutations',
    })

    gdc['nih_cnv_loss'] = gdc['cohort_cnv_homozygous_deletion_cases_percentage']

    for col in ['nih_ssm_in_cohort', 'nih_ssm_across_gdc', 'nih_cnv_gain', 'nih_cnv_loss']:
        gdc[col] = gdc[col] / 100.0

    gdc = gdc[['ensembl', 'nih_ssm_in_cohort', 'nih_ssm_across_gdc',
               'nih_cnv_gain', 'nih_cnv_loss', 'nih_tot_mutations']]

    return gdc

In [7]:
import os

def Runnode2vec(filepath):
    df = pd.read_csv(filepath, sep="\t", header=None, names=["source", "target", "weight"])
    df["weight"] /= 1000  # normalise STRING combined_score to [0,1]

    G = nx.from_pandas_edgelist(df, "source", "target", ["weight"], create_using=nx.Graph())

    # Match worker count to actual available CPUs instead of hardcoding 4.
    # Requesting more worker processes than physical cores doesn't parallelise
    # anything extra -- each worker gets its own copy of the graph/alias
    # tables in memory, so oversubscribing workers on a low-core machine
    # burns RAM for no speed benefit and can itself contribute to crashes.
    n_workers = max(1, min(4, os.cpu_count() or 1))

    node2vec = Node2Vec(
        G, dimensions=128, walk_length=60, num_walks=15,
        workers=n_workers, p=1, q=0.5,
        quiet=True,          # skip per-walk progress bar overhead
    )
    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    emd = pd.DataFrame([model.wv[str(node)] for node in G.nodes()], index=G.nodes())
    emd.columns = [f'network_{i}' for i in range(emd.shape[1])]

    return emd.reset_index().rename(columns={"index": "ensembl"})

## Run pipeline

In [8]:
hpa = ImportHPA()
hpa

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,-0.017757,-0.026070,0.011761,-0.002426,-0.019913,0.009452,0.023949,0.006893,-0.003688,-0.007344
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,-0.013362,-0.010440,0.003159,0.004360,-0.014105,0.012614,0.011542,-0.001288,-0.009632,-0.000528
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.057882,0.015543,0.025354,-0.063970,-0.021730,0.013750,0.019755,-0.022991,-0.020344,-0.023098
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,0.011027,0.027119,-0.025234,-0.004661,-0.018219,-0.016444,-0.022047,-0.011655,0.017283,0.014506
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,0.010409,0.046441,0.086334,0.107279,0.013497,-0.026416,0.014785,0.078126,0.122070,-0.072241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14011,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,0.002179,-0.004050,0.004108,0.000805,-0.001073,-0.003030,-0.004864,0.001746,-0.005832,0.009135
14012,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.017844,0.002500,0.007794,-0.023858,0.003625,-0.009655,-0.004265,-0.012930,0.002998,-0.001015
14013,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.013388,0.001719,0.003715,-0.020730,-0.022674,-0.000576,-0.018999,-0.008440,-0.011515,0.013988
14014,ENSG00000074755,0.761028,0.736261,0.581626,0.452769,1.144209,-0.162387,0.335192,0.342591,0.245857,...,0.001624,-0.011565,0.006311,-0.004188,-0.008739,0.001338,0.004442,0.002833,-0.007882,0.001370


In [9]:
gdc=ImportGDC()
gdc

,ensembl,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000155657,0.5081,0.2901,0.2172,0.0,825
1,ENSG00000141510,0.5027,0.3137,0.0489,0.0,208
2,ENSG00000181143,0.4436,0.1849,0.0333,0.0,505
3,ENSG00000198626,0.4204,0.1305,0.3933,0.0,434
4,ENSG00000164796,0.4132,0.1315,0.3014,0.0,409
...,...,...,...,...,...,...
20363,ENSG00000288660,0.0000,0.0000,0.2270,0.0,0
20364,ENSG00000288669,0.0000,0.0000,0.0313,0.0,0
20365,ENSG00000288671,0.0000,0.0000,0.1311,0.0,0
20366,ENSG00000288674,0.0000,0.0000,0.4031,0.0,0


In [10]:
el = ImportSTRING()

print(f'Edges before filtering: {len(el):,}')
el = el.loc[el['combined_score'] > 700].reset_index(drop=True)
print(f'Edges after removing score >= 400: {len(el):,}')

el

Original Size:  13715404
After Dropping Duplicates:  6857702 (- 6857702 )
After Merging:  6857702 (- 0 )

NaN values per Column: 0 0
New edge list size: 6857702 ( - 0 )
Edges before filtering: 6,857,702
Edges after removing score >= 400: 236,000


,protein1,protein2,combined_score
0,ENSG00000004059,ENSG00000072818,825
1,ENSG00000004059,ENSG00000122218,718
2,ENSG00000004059,ENSG00000090565,952
3,ENSG00000004059,ENSG00000184432,752
4,ENSG00000004059,ENSG00000105669,795
...,...,...,...
235995,ENSG00000143933,ENSG00000070808,962
235996,ENSG00000143933,ENSG00000145335,918
235997,ENSG00000162434,ENSG00000051382,933
235998,ENSG00000070808,ENSG00000121281,707


In [11]:
master = hpa
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,-0.017757,-0.026070,0.011761,-0.002426,-0.019913,0.009452,0.023949,0.006893,-0.003688,-0.007344
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,-0.013362,-0.010440,0.003159,0.004360,-0.014105,0.012614,0.011542,-0.001288,-0.009632,-0.000528
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.057882,0.015543,0.025354,-0.063970,-0.021730,0.013750,0.019755,-0.022991,-0.020344,-0.023098
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,0.011027,0.027119,-0.025234,-0.004661,-0.018219,-0.016444,-0.022047,-0.011655,0.017283,0.014506
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,0.010409,0.046441,0.086334,0.107279,0.013497,-0.026416,0.014785,0.078126,0.122070,-0.072241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14011,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,0.002179,-0.004050,0.004108,0.000805,-0.001073,-0.003030,-0.004864,0.001746,-0.005832,0.009135
14012,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.017844,0.002500,0.007794,-0.023858,0.003625,-0.009655,-0.004265,-0.012930,0.002998,-0.001015
14013,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.013388,0.001719,0.003715,-0.020730,-0.022674,-0.000576,-0.018999,-0.008440,-0.011515,0.013988
14014,ENSG00000074755,0.761028,0.736261,0.581626,0.452769,1.144209,-0.162387,0.335192,0.342591,0.245857,...,0.001624,-0.011565,0.006311,-0.004188,-0.008739,0.001338,0.004442,0.002833,-0.007882,0.001370


In [12]:
master = master.merge(gdc, on='ensembl')
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,0.009452,0.023949,0.006893,-0.003688,-0.007344,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,0.012614,0.011542,-0.001288,-0.009632,-0.000528,0.0483,0.0238,0.1703,0.0,30
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.013750,0.019755,-0.022991,-0.020344,-0.023098,0.0000,0.0066,0.0881,0.0,0
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.016444,-0.022047,-0.011655,0.017283,0.014506,0.0018,0.0072,0.1840,0.0,1
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,-0.026416,0.014785,0.078126,0.122070,-0.072241,0.0268,0.0101,0.1683,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13894,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.003030,-0.004864,0.001746,-0.005832,0.009135,0.0143,0.0101,0.1898,0.0,10
13895,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.009655,-0.004265,-0.012930,0.002998,-0.001015,0.0197,0.0093,0.1722,0.0,11
13896,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.000576,-0.018999,-0.008440,-0.011515,0.013988,0.0125,0.0080,0.2505,0.0,8
13897,ENSG00000074755,0.761028,0.736261,0.581626,0.452769,1.144209,-0.162387,0.335192,0.342591,0.245857,...,0.001338,0.004442,0.002833,-0.007882,0.001370,0.0411,0.0260,0.0548,0.0,24


In [13]:
list_of_dropped_genes=pd.read_csv("ESCC_gene_x_celltype_mean_expression.csv")
master_590 = master[
    master["ensembl"].isin(list_of_dropped_genes["ensembl"])
]
master=master_590
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,0.009452,0.023949,0.006893,-0.003688,-0.007344,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,0.012614,0.011542,-0.001288,-0.009632,-0.000528,0.0483,0.0238,0.1703,0.0,30
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.013750,0.019755,-0.022991,-0.020344,-0.023098,0.0000,0.0066,0.0881,0.0,0
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.016444,-0.022047,-0.011655,0.017283,0.014506,0.0018,0.0072,0.1840,0.0,1
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,-0.026416,0.014785,0.078126,0.122070,-0.072241,0.0268,0.0101,0.1683,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13892,ENSG00000198205,1.240307,1.425246,0.755020,-0.038078,0.663685,-0.156014,-0.025512,-0.149714,0.010578,...,-0.003208,-0.002641,0.001965,-0.001732,0.009616,0.0179,0.0125,0.0000,0.0,10
13894,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.003030,-0.004864,0.001746,-0.005832,0.009135,0.0143,0.0101,0.1898,0.0,10
13895,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.009655,-0.004265,-0.012930,0.002998,-0.001015,0.0197,0.0093,0.1722,0.0,11
13896,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.000576,-0.018999,-0.008440,-0.011515,0.013988,0.0125,0.0080,0.2505,0.0,8


In [14]:
master = master.reset_index(drop=True)
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,0.009452,0.023949,0.006893,-0.003688,-0.007344,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,0.012614,0.011542,-0.001288,-0.009632,-0.000528,0.0483,0.0238,0.1703,0.0,30
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.013750,0.019755,-0.022991,-0.020344,-0.023098,0.0000,0.0066,0.0881,0.0,0
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.016444,-0.022047,-0.011655,0.017283,0.014506,0.0018,0.0072,0.1840,0.0,1
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,-0.026416,0.014785,0.078126,0.122070,-0.072241,0.0268,0.0101,0.1683,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10530,ENSG00000198205,1.240307,1.425246,0.755020,-0.038078,0.663685,-0.156014,-0.025512,-0.149714,0.010578,...,-0.003208,-0.002641,0.001965,-0.001732,0.009616,0.0179,0.0125,0.0000,0.0,10
10531,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.003030,-0.004864,0.001746,-0.005832,0.009135,0.0143,0.0101,0.1898,0.0,10
10532,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.009655,-0.004265,-0.012930,0.002998,-0.001015,0.0197,0.0093,0.1722,0.0,11
10533,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.000576,-0.018999,-0.008440,-0.011515,0.013988,0.0125,0.0080,0.2505,0.0,8


In [15]:
el_allgenes = pd.concat([el['protein1'], el['protein2']]).drop_duplicates()
master = master.loc[master['ensembl'].isin(el_allgenes)]
print(f'Genes after STRING intersection: {len(master)}')
master

Genes after STRING intersection: 10535


,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,0.009452,0.023949,0.006893,-0.003688,-0.007344,0.0394,0.0247,0.1722,0.0,23
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,0.012614,0.011542,-0.001288,-0.009632,-0.000528,0.0483,0.0238,0.1703,0.0,30
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,0.013750,0.019755,-0.022991,-0.020344,-0.023098,0.0000,0.0066,0.0881,0.0,0
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.016444,-0.022047,-0.011655,0.017283,0.014506,0.0018,0.0072,0.1840,0.0,1
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,-0.026416,0.014785,0.078126,0.122070,-0.072241,0.0268,0.0101,0.1683,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10530,ENSG00000198205,1.240307,1.425246,0.755020,-0.038078,0.663685,-0.156014,-0.025512,-0.149714,0.010578,...,-0.003208,-0.002641,0.001965,-0.001732,0.009616,0.0179,0.0125,0.0000,0.0,10
10531,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.003030,-0.004864,0.001746,-0.005832,0.009135,0.0143,0.0101,0.1898,0.0,10
10532,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,-0.009655,-0.004265,-0.012930,0.002998,-0.001015,0.0197,0.0093,0.1722,0.0,11
10533,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.000576,-0.018999,-0.008440,-0.011515,0.013988,0.0125,0.0080,0.2505,0.0,8


In [16]:
el_intersect = (
    el.iloc[:, :3]
    .merge(master["ensembl"], right_on="ensembl", left_on='protein1')
    .drop("ensembl", axis=1)
)
el_intersect = (
    el_intersect
    .merge(master["ensembl"], right_on="ensembl", left_on='protein2')
    .drop("ensembl", axis=1)
    .rename(columns={'protein1': 'gene1', 'protein2': 'gene2'})
)

el = el_intersect.merge(el, right_on=['protein1', 'protein2'], left_on=['gene1', 'gene2']).drop(['protein1', 'protein2'], axis=1)
el

,gene1,gene2,combined_score_x,combined_score_y
0,ENSG00000004059,ENSG00000072818,825,825
1,ENSG00000004059,ENSG00000122218,718,718
2,ENSG00000004059,ENSG00000090565,952,952
3,ENSG00000004059,ENSG00000184432,752,752
4,ENSG00000004059,ENSG00000105669,795,795
...,...,...,...,...
152627,ENSG00000198668,ENSG00000145335,973,973
152628,ENSG00000274211,ENSG00000162434,786,786
152629,ENSG00000143933,ENSG00000145335,918,918
152630,ENSG00000162434,ENSG00000051382,933,933


In [17]:
el[['gene1', 'gene2', 'combined_score_x']].to_csv(
    '10k_esca.edg', index=False, header=False, sep='\t'
)

In [18]:
df = pd.read_csv(
    "10k_esca.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [19]:
network = Runnode2vec("10k_esca.edg")
master = master.merge(network, on='ensembl')
master.to_csv("node_node2vec_data_latest.csv", index=None)
master

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,-0.331405,-0.088331,0.646216,0.045284,0.142957,0.049476,-0.307549,0.267607,-0.011635,-0.407689
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,-0.384218,0.287173,-0.534908,0.145208,-0.362811,0.837559,-0.304619,-0.144369,-0.043333,-0.247883
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,-0.024244,0.018693,-0.024622,-0.189089,0.191551,1.371162,0.331431,0.070105,0.142604,0.012307
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.028360,-0.252449,-0.137181,0.126925,-0.156854,-0.006608,0.109685,0.412189,-0.302970,-0.095583
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,0.197927,-0.376011,-0.203497,-0.286824,0.352305,0.335333,-0.008814,-0.308114,0.040001,0.439726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10527,ENSG00000198205,1.240307,1.425246,0.755020,-0.038078,0.663685,-0.156014,-0.025512,-0.149714,0.010578,...,-0.228147,-0.473521,0.714811,-0.674046,0.358543,0.061893,0.061073,0.151215,-0.273580,-0.138739
10528,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.248448,-0.470064,0.638149,-0.632153,0.447876,0.166523,-0.140376,0.327379,-0.380241,-0.043763
10529,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,0.382491,-0.268029,-0.331272,-0.017037,-0.026989,0.340596,0.165021,0.112965,-0.077227,0.031839
10530,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.300363,0.383645,-0.403363,-0.299021,-0.030125,0.240254,-0.137387,0.533011,0.104956,0.157842


In [20]:
master.to_csv("node_networkfeatures_10k_cnv.csv", index=False)

In [21]:
df = pd.read_csv(
    "10k_esca.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [22]:
edge_array = df[["source", "target", "weight"]].to_numpy()
np.save("10k_esca.npy", edge_array, allow_pickle=True)
print("Saved edge_list_latest1.npy with shape:", edge_array.shape)

loaded = np.load("10k_esca.npy", allow_pickle=True)
print(loaded[:5])

Saved edge_list_latest1.npy with shape: (152632, 3)
[['ENSG00000004059' 'ENSG00000072818' 825]
 ['ENSG00000004059' 'ENSG00000122218' 718]
 ['ENSG00000004059' 'ENSG00000090565' 952]
 ['ENSG00000004059' 'ENSG00000184432' 752]
 ['ENSG00000004059' 'ENSG00000105669' 795]]


In [23]:
master=pd.read_csv("node_networkfeatures_10k_cnv.csv")
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000175899,0.423392,-0.140290,0.080555,0.085041,-0.201826,1.727767,-0.198632,0.392170,0.054074,...,-0.331405,-0.088331,0.646216,0.045284,0.142957,0.049476,-0.307549,0.267607,-0.011635,-0.407689
1,ENSG00000166535,0.927238,-0.115654,0.115089,0.046566,-0.338453,0.024291,-0.411618,-0.287169,-0.480022,...,-0.384218,0.287173,-0.534908,0.145208,-0.362811,0.837558,-0.304619,-0.144369,-0.043333,-0.247884
2,ENSG00000128274,0.684248,-0.656108,-1.028539,-0.463610,1.425009,-0.112268,0.217956,-0.040394,0.106205,...,-0.024244,0.018693,-0.024622,-0.189089,0.191551,1.371162,0.331431,0.070105,0.142604,0.012307
3,ENSG00000094914,2.396667,-0.836030,0.242969,0.395142,-0.271862,-1.073339,0.356226,0.212867,0.739628,...,-0.028360,-0.252449,-0.137181,0.126925,-0.156854,-0.006608,0.109685,0.412189,-0.302970,-0.095583
4,ENSG00000081760,0.626831,-0.581950,-0.886576,-0.060819,1.208408,-0.018577,0.088175,-0.108904,-0.075075,...,0.197927,-0.376011,-0.203497,-0.286824,0.352305,0.335333,-0.008814,-0.308115,0.040001,0.439726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10527,ENSG00000198205,1.240307,1.425246,0.755020,-0.038078,0.663685,-0.156014,-0.025512,-0.149714,0.010578,...,-0.228147,-0.473521,0.714811,-0.674046,0.358543,0.061893,0.061073,0.151215,-0.273580,-0.138739
10528,ENSG00000070476,0.987418,1.113407,0.672241,-0.039499,0.581539,0.051887,-0.184555,-0.871890,0.063558,...,-0.248448,-0.470064,0.638149,-0.632153,0.447876,0.166523,-0.140376,0.327379,-0.380241,-0.043763
10529,ENSG00000162378,0.852646,0.258676,-0.414860,0.270404,-0.078690,-0.011411,-0.484232,-0.367752,-0.664468,...,0.382491,-0.268029,-0.331272,-0.017037,-0.026989,0.340596,0.165021,0.112965,-0.077227,0.031839
10530,ENSG00000159840,0.957870,0.201387,-0.388779,0.335228,-0.384886,0.650683,-0.472806,-0.265029,0.049278,...,-0.300363,0.383645,-0.403363,-0.299021,-0.030125,0.240254,-0.137387,0.533011,0.104956,0.157842


In [24]:
dgn = ImportDGN()
dgn = dgn.loc[dgn['ensembl'].isin(master['ensembl'])]
print(f'LIHC GDA genes overlapping final master: {len(dgn)}')
dgn

LIHC GDA genes overlapping final master: 79


,ensembl,gda_score
0,ENSG00000146648,1
1,ENSG00000141510,1
2,ENSG00000168036,1
3,ENSG00000124762,1
4,ENSG00000147889,1
...,...,...
94,ENSG00000148516,1
95,ENSG00000105329,1
97,ENSG00000122691,1
100,ENSG00000092820,1


In [25]:
master["gda_score"] = np.nan
master.loc[master["ensembl"].isin(dgn["ensembl"]), "gda_score"] = 1

num_ones = (master["gda_score"] == 1).sum()
print("Number of positive (gda_score=1) genes:", num_ones)

Number of positive (gda_score=1) genes: 79


In [26]:
master.to_csv("200_10k_cnv.csv", index=None)
print('Saved: 200_node_network_features_latest.csv')

Saved: 200_node_network_features_latest.csv
